In [11]:
using Lux, DifferentialEquations, Optimization, OptimizationOptimisers, SciMLSensitivity
using Random, ComponentArrays, Statistics, Plots, CSV, DataFrames, Zygote


In [13]:
# 1. Load the data
file_path = "c:/Users/ADMIN/Downloads/Neural_Spiking_Dynamics/notebooks/1_data_generation/single_spike_noisy_data.csv"
HH_data = CSV.read(file_path, DataFrame)

# 2. Extract the relevant columns in order
df_ordered = HH_data[:, [:timestamp, :V, :n, :m, :h]]

# 3. Create the training arrays (t_train and data_train)
t_train = Float32.(df_ordered.timestamp)
data_train = Float32.(Matrix(df_ordered[:, [:V, :n, :m, :h]])')


4×1469 Matrix{Float32}:
 -65.0   -64.8363     -64.7979    …  -57.3823    -57.4143    -57.4497
   0.6     0.599784     0.599864       0.453623    0.453502    0.453245
   0.05    0.0502886    0.050523       0.112056    0.112553    0.112654
   0.32    0.320059     0.32014        0.395085    0.395061    0.39517

In [14]:
function hh_true_dynamics(u, p, t)
    V, n, m, h = u
    # Constants
    C_m, E_Na, E_K, E_L = 1.0f0, 50.0f0, -77.0f0, -54.4f0
    g_Na, g_K, g_L, I_ext = 120.0f0, 36.0f0, 0.3f0, 10.0f0

    # Rate functions
    α_n = 0.01f0 * (V + 55) / (1 - exp(-(V + 55) / 10))
    β_n = 0.125f0 * exp(-(V + 65) / 80)
    α_m = 0.1f0 * (V + 40) / (1 - exp(-(V + 40) / 10))
    β_m = 4.0f0 * exp(-(V + 65) / 18)
    α_h = 0.07f0 * exp(-(V + 65) / 20)
    β_h = 1.0f0 / (1 + exp(-(V + 35) / 10))

    # Currents
    I_Na = g_Na * m^3 * h * (V - E_Na)
    I_K  = g_K * n^4 * (V - E_K)
    I_L  = g_L * (V - E_L)

    dV = (I_ext - I_Na - I_K - I_L) / C_m
    dn = α_n * (1 - n) - β_n * n
    dm = α_m * (1 - m) - β_m * m
    dh = α_h * (1 - h) - β_h * h

    return [dV, dn, dm, dh]
end

hh_true_dynamics (generic function with 1 method)

In [15]:
u0 = [-65.0f0, 0.05f0, 0.6f0, 0.32f0]
tspan = (0.0f0, 30.0f0)
prob_true = ODEProblem(hh_true_dynamics, u0, tspan)

ODEProblem with uType Vector{Float32} and tType Float32. In-place: false
Non-trivial mass matrix: false
timespan: (0.0f0, 30.0f0)
u0: 4-element Vector{Float32}:
 -65.0
   0.05
   0.6
   0.32

In [ ]:
# ====================================================================
# CELL 3: STRUCTURAL SETUP & PARAMETER CONTAINER
# ====================================================================
rng = Random.default_rng()
Random.seed!(rng, 42) # Set seed for reproducibility

nn_model = Lux.Chain(
    Lux.Dense(4 => 32, tanh),
    Lux.Dense(32 => 32, tanh),
    Lux.Dense(32 => 1) 
)
p_nn, st_nn = Lux.setup(rng, nn_model)

# Flatten parameters and structural states safely for Optimization.jl
p_init = ComponentArray((nn = p_nn, st = st_nn))

println("✓ Cell 3 Complete: Neural Network initialized and wrapped into ComponentArray.")

((layer_1 = (weight = Float32[1.430542 -1.2234261 -0.31329527 -0.5285851; -1.0636693 -1.1448612 0.7424008 -0.7000711; … ; 0.22226545 0.3665172 -0.0051515894 -0.41045108; -0.20999506 -0.008929766 -1.2735714 -0.85856485], bias = Float32[-0.13025641, -0.11415768, -0.15128064, -0.29241908, 0.28074372, 0.056691945, 0.024069786, -0.33957332, -0.23938507, -0.46831155  …  0.1696577, 0.32883263, 0.18119794, 0.2950155, 0.08643162, -0.42653906, 0.1511097, 0.278377, -0.36365086, 0.11294049]), layer_2 = (weight = Float32[-0.30600992 0.00076772174 … 0.21083336 -0.060751792; 0.0501725 -0.44111088 … -0.16306508 0.36051148; … ; -0.21903697 0.38901544 … -0.36754465 0.01679948; -0.409722 -0.48108596 … -0.46207803 -0.35052153], bias = Float32[0.033687197, -0.11234678, 0.11817062, -0.1535161, -0.1621318, 0.09666321, -0.08353195, 0.059416078, -0.0874403, 0.16021974  …  0.09585509, -0.07839611, -0.11114868, -0.15976909, 0.020422382, -0.15192759, -0.12256542, 0.12980819, 0.086603895, 0.13614663]), layer_3 = (

In [ ]:
# ====================================================================
# CELL 4: FIXED HYBRID DYNAMICS, PREDICTION & LOSS PIPELINE
# ====================================================================

function ude_dynamics_fixed(u, p, t)
    V, n, m, h = u
    C_m, E_K, E_L = 1.0f0, -77.0f0, -54.4f0
    g_K, g_L, I_ext = 36.0f0, 0.3f0, 10.0f0

    I_K = g_K * n^4 * (V - E_K)
    I_L = g_L * (V - E_L)
    
    α_n = 0.01f0 * (V + 55) / (1 - exp(-(V + 55) / 10))
    β_n = 0.125f0 * exp(-(V + 65) / 80)
    α_m = 0.1f0 * (V + 40) / (1 - exp(-(V + 40) / 10))
    β_m = 4.0f0 * exp(-(V + 65) / 18)
    α_h = 0.07f0 * exp(-(V + 65) / 20)
    β_h = 1.0f0 / (1 + exp(-(V + 35) / 10))

    dn = α_n * (1 - n) - β_n * n
    dm = α_m * (1 - m) - β_m * m
    dh = α_h * (1 - h) - β_h * h

    # Safe type-stable evaluation for Zygote tracking
    u_spatial = reshape(u, :, 1)
    nn_out, _ = nn_model(u_spatial, p.nn, p.st)
    du_missing = nn_out[1]
    
    dV = ((I_ext - I_K - I_L) / C_m) + du_missing
    return [dV, dn, dm, dh]
end

prob_ude = ODEProblem(ude_dynamics_fixed, u0, tspan, p_init)

function predict_ude(θ)
    sol = solve(prob_ude, Tsit5(), p=θ, saveat=t_train, 
                sensealg=InterpolatingAdjoint(autojacvec=ZygoteVJP()),
                abstol=1e-6, reltol=1e-6)
    return Array(sol)
end

function loss_function(θ, _)
    pred = predict_ude(θ)
    loss = mean(abs2, pred .- data_train)
    return loss, pred 
end

println("✓ Cell 4 Complete: Hybrid ODE problem and differentiable loss function defined.")

ude_dynamics (generic function with 1 method)

In [ ]:
# ====================================================================
# CELL 5: TWO-STAGE OPTIMIZATION TRAINING LOOP
# ====================================================================
adtype = Optimization.AutoZygote()
optf = OptimizationFunction((θ, p) -> loss_function(θ, p)[1], adtype)
optprob = OptimizationProblem(optf, p_init)

callback = function (p, l)
    println("Current MSE Loss value: ", round(l, digits=6))
    return false
end

println("--- Starting Stage 1: Adam (Coarse Global Search) ---")
res1 = solve(optprob, Adam(0.01), maxiters=200, callback=callback)

println("\n--- Starting Stage 2: BFGS (Local Fine-Tuning) ---")
optprob2 = remake(optprob, u0=res1.u)
res2 = solve(optprob2, BFGS(linesearch=LineSearches.BackTracking()), maxiters=100, callback=callback)
p_trained = res2.u

println("\n✓ Cell 5 Complete: Training completed successfully!")

predict_ude (generic function with 1 method)

In [ ]:
# ====================================================================
# CELL 6: VALIDATION & PLOTTING MISSING PHYSICS
# ====================================================================
p_trained_nn = p_trained.nn
st_trained_nn = p_trained.st

V_range = data_train[1, :]
extracted_nn_current = zeros(Float32, length(V_range))

for i in 1:length(V_range)
    state_vector = data_train[:, i]
    nn_out, _ = nn_model(reshape(state_vector, :, 1), p_trained_nn, st_trained_nn)
    extracted_nn_current[i] = nn_out[1]
end

true_missing = zeros(Float32, length(V_range))
for i in 1:length(V_range)
    V, n, m, h = data_train[:, i]
    I_Na = 120.0f0 * (m^3) * h * (V - 50.0f0)
    true_missing[i] = -I_Na / 1.0f0
end

# Generate Plot
p1 = plot(t_train, data_train[1,:], label="True Voltage Data", color=:black, lw=2)
plot!(p1, t_train, predict_ude(p_trained)[1,:], label="UDE Prediction", color=:red, linestyle=:dash, title="State Trajectory Match")

p2 = plot(t_train, true_missing, label="True Na+ Dynamics (-I_Na)", color=:blue, lw=2)
plot!(p2, t_train, extracted_nn_current, label="Discovered Term (NN)", color=:orange, linestyle=:dash, title="Missing Physics Reconstruction")

plot(p1, p2, layout=(2,1), size=(800,600))

loss_function (generic function with 1 method)